# Step 2: BigQuery Property Graph 설계 및 GQL 질의 체계
## Google Cloud CE를 위한 온톨로지 스타터팩 핸즈온 시리즈 (Part 2/3)

- **학습 목표**: 관계형 DW 테이블을 외부 그래프 DB(Neo4j 등)로 복제하지 않고, BigQuery 원장 테이블을 물리 그래프 노드로 직접 승격하는 **2-Layer 안티 리던던시(Anti-Redundancy)** 원리와 **BigQuery ISO SQL:2023 Property Graph DDL**, 그리고 **GQL 다중 홉 탐색**을 실습합니다.
- **연계 표준 방법론**: [`Stage 3: BigQuery Property Graph 엔지니어링 및 GQL 질의 체계 세부 실행 명세서`](../methodology/03_property_graph_and_gql_ko.md)
- **핵심 안내 (The Remote MCP Gap)**: 공식 Google Cloud BigQuery Remote MCP의 `execute_sql_readonly`는 GQL을 미지원하므로, 프로덕션 환경에서 왜 Tier 3 Domain MCP 계층이 필수적인지 학습합니다.
- **외부 종속성**: **0개 (파이썬 표준 라이브러리 완결)**

---

### 1. 왜 기존 지식 그래프는 DW 복제 병폐로 실패하는가? (Concept)
기존 엔터프라이즈 그래프 프로젝트는 이미 빅쿼리에 잘 정제되어 있는 수천만 행의 정형 원장 테이블(`devices`, `plans`, `subsidies`)을 별도의 그래프 DB나 트리플 스토어(`Subject-Predicate-Object`)로 중복 복제하는 치명적 실수를 범합니다:
1. **데이터 정합성 파괴**: DW 원장이 갱신될 때 그래프 동기화 지연(ETL Lag)으로 가격/지원금 불일치 발생.
2. **저장소 및 비용 80% 낭비**: 동일 데이터를 이중 저장하고 비싼 그래프 DB 라이선스 비용 지출.

**아키텍처 해법: BigQuery SQL:2023 Property Graph & 2-Layer 안티 리던던시**:
- **Direct Promotion (원장 노드 직접 승격)**: 빅쿼리 DW 테이블 자체를 `NODE TABLES`로 선언 (Zero-Copy 가상화).
- **Redundant Entity Pruning**: 비정형 텍스트에서 추출된 단말/요금제 엔티티는 중복 노드로 생성하지 않고 전량 제거(Pruning). 오직 증거 청크(`DocChunk`)와 규칙(`DomainRule`)만 노드로 선언.
- **Semantic Bridges (시맨틱 브릿지)**: 정형 DW 원장과 비정형 규정 청크를 `APPLIES_RULE`, `GROUNDED_IN` 엣지로 직접 연결.


### 2. Step 2 그래프 토폴로지 및 GQL 순회 플로우 (Flow Architecture)

```mermaid
flowchart LR
    subgraph DW ["정형 DW 마스터 테이블 (Direct Promotion)"]
        D["DeviceNode<br/>(dw_device_master)"]
        P["PlanNode<br/>(dw_plan_master)"]
        S["SubsidyNode<br/>(dw_subsidy_policy_master)"]
        S -->|COVERS_DEVICE| D
        S -->|REQUIRES_PLAN| P
    end

    subgraph Unstructured ["비정형 지식 잔존 노드 (Pruned & Grounded)"]
        R["DomainRuleNode<br/>(180일 식스플랜 규칙)"]
        C["DocChunkNode<br/>(DOC_CHUNK_002 원천 증거)"]
        R -->|GROUNDED_IN| C
    end

    subgraph SemanticBridge ["시맨틱 크로스도메인 브릿지"]
        R -.->|APPLIES_RULE| P
    end

    classDef dw fill:#e8f0fe,stroke:#1a73e8,stroke-width:2px;
    classDef un fill:#f3e8fd,stroke:#9334e8,stroke-width:2px;
    classDef br fill:#e6f4ea,stroke:#137333,stroke-width:2px;

    class D,P,S dw;
    class R,C un;
```


### [실습 2.1] 빅쿼리 정형 DW 원장 테이블 프로파일링


In [1]:
import os, sys, csv

starter_root = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
tables_dir = os.path.join(starter_root, "data", "input", "03_tables")
tables = ["devices.csv", "plans.csv", "subsidies.csv"]

print("=== [BigQuery DW 물리 마스터 원장 프로파일링] ===")
for t in tables:
    tpath = os.path.join(tables_dir, t)
    with open(tpath, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader)
        rows = list(reader)
    tname = t.replace(".csv", "")
    print(f"\n✔ 테이블명: `dw_{tname}_master` | 행수: {len(rows)} | 컬럼수: {len(header)}")
    print(f"  컬럼 스키마: {', '.join(header)}")
    print(f"  샘플 레코드: {rows[0]}")


=== [BigQuery DW 물리 마스터 원장 프로파일링] ===

✔ 테이블명: `dw_devices_master` | 행수: 4 | 컬럼수: 5
  컬럼 스키마: device_id, model_code, pet_name, storage, factory_price
  샘플 레코드: ['DEV-S26U-256', 'SM-S938N-256', 'Galaxy S26 Ultra', '256GB', '1698400']

✔ 테이블명: `dw_plans_master` | 행수: 6 | 컬럼수: 5
  컬럼 스키마: plan_id, plan_name, network_type, monthly_fee, is_baseline_tier
  샘플 레코드: ['PLAN-5G-130', '5G Premier Super', '5G', '130000', 'true']

✔ 테이블명: `dw_subsidies_master` | 행수: 5 | 컬럼수: 7
  컬럼 스키마: policy_id, device_id, plan_id, basic_subsidy, distribution_subsidy, effective_start_date, effective_end_date
  샘플 레코드: ['POL-2026-001', 'DEV-S26U-256', 'PLAN-5G-130', '500000', '75000', '2026-01-01', '2026-12-31']


### [실습 2.2] BigQuery ISO SQL:2023 Property Graph DDL 자동 합성 (`pipeline.graph_builder`)


In [2]:
from pipeline.graph_builder import build_graph

topology = build_graph(project_id="my-gcp-project", dataset_id="telecom_ontology", output_dir=os.path.join(starter_root, "data", "output"))

ddl_path = os.path.join(starter_root, "data", "output", "create_property_graph.sql")
with open(ddl_path, "r", encoding="utf-8") as f:
    ddl_sql = f.read()

print("=== [생성된 BigQuery SQL:2023 Property Graph DDL 전문 발췌] ===")
print(ddl_sql[:950] + "\n... (전체 72라인 중 상위 950자 표출)\n")

print("=== [2-Layer 안티 리던던시 토폴로지 검증] ===")
print(f"그래프 식별자: {topology['graph_name']}")
print(f"정형 DW 노드 직접 승격: {topology['structured_dw_nodes']} (3개 테이블)")
print(f"비정형 필수 노드: {topology['unstructured_pruned_nodes']} (2개 노드)")
print(f"시맨틱 엣지 테이블: {topology['edge_tables']}")
print(f"중복 엔티티 노드 저장수: {topology['metrics']['duplicate_entity_nodes']} (완전 0건 달성)")


[GRAPH BUILDER SUCCESS] DDL generated -> /Users/seanjung/Downloads/Uplus Ontology/agentic-ontology-starter/data/output/create_property_graph.sql, Topology -> /Users/seanjung/Downloads/Uplus Ontology/agentic-ontology-starter/data/output/graph_topology.json
=== [생성된 BigQuery SQL:2023 Property Graph DDL 전문 발췌] ===
-- BigQuery SQL:2023 Property Graph DDL Definition
-- Architecture: 2-Layer Anti-Redundancy (Zero duplicate entity storage)
CREATE OR REPLACE PROPERTY GRAPH `my-gcp-project.telecom_ontology.ENTERPRISE_SUBSIDY_GRAPH`
  NODE TABLES (
    -- [Track B & C] Structured DW Master Tables Promoted Directly to Graph Nodes
    `my-gcp-project.telecom_ontology.dw_device_master`
       KEY (device_id)
       LABEL DeviceNode
       PROPERTIES (device_id, model_code, pet_name, storage, factory_price),
    `my-gcp-project.telecom_ontology.dw_plan_master`
       KEY (plan_id)
       LABEL PlanNode
       PROPERTIES (plan_id, plan_name, network_type, monthly_fee, is_baseline_tier),
    `my-gcp-p

### [실습 2.3] BigQuery GQL `GRAPH_TABLE()` 다중 홉 탐색 및 Remote MCP Gap 분석
BigQuery 표준 GQL 문법을 사용하여 `DeviceNode` ➔ `SubsidyNode` ➔ `PlanNode` ➔ `DomainRuleNode` ➔ `DocChunkNode`의 4-Hop 경로를 단일 쿼리로 탐색합니다.


In [3]:
# BigQuery GQL 질의 구문 예시
sample_gql = """
SELECT *
FROM GRAPH_TABLE(
  `my-gcp-project.telecom_ontology.ENTERPRISE_SUBSIDY_GRAPH`
  MATCH (device:DeviceNode)<-[:COVERS_DEVICE]-(subsidy:SubsidyNode)-[:REQUIRES_PLAN]->(plan:PlanNode),
        (rule:DomainRuleNode)-[:APPLIES_RULE]->(plan),
        (rule)-[:GROUNDED_IN]->(chunk:DocChunkNode)
  WHERE device.pet_name = 'Galaxy S26 Ultra'
    AND plan.monthly_fee >= 80000
  COLUMNS (
    device.pet_name AS device_model,
    device.factory_price,
    subsidy.basic_subsidy,
    subsidy.distribution_subsidy,
    plan.plan_name,
    plan.monthly_fee,
    rule.rule_name,
    chunk.hash_anchor
  )
)
"""

print("=== [BigQuery SQL:2023 GQL 4-Hop Multi-Match Query] ===")
print(sample_gql.strip())
print("\n[GCP CE 핵심 아키텍처 포인트 - The Remote MCP Gap]:")
print("1. 구글 공식 BigQuery Remote MCP(execute_sql_readonly)는 GQL 문법을 파싱하지 못함.")
print("2. 따라서 AI 에이전트가 GQL 쿼리를 날릴 수 있도록 Step 3의 'Tier 3 Domain MCP'가 필수적임.")


=== [BigQuery SQL:2023 GQL 4-Hop Multi-Match Query] ===
SELECT *
FROM GRAPH_TABLE(
  `my-gcp-project.telecom_ontology.ENTERPRISE_SUBSIDY_GRAPH`
  MATCH (device:DeviceNode)<-[:COVERS_DEVICE]-(subsidy:SubsidyNode)-[:REQUIRES_PLAN]->(plan:PlanNode),
        (rule:DomainRuleNode)-[:APPLIES_RULE]->(plan),
        (rule)-[:GROUNDED_IN]->(chunk:DocChunkNode)
  WHERE device.pet_name = 'Galaxy S26 Ultra'
    AND plan.monthly_fee >= 80000
  COLUMNS (
    device.pet_name AS device_model,
    device.factory_price,
    subsidy.basic_subsidy,
    subsidy.distribution_subsidy,
    plan.plan_name,
    plan.monthly_fee,
    rule.rule_name,
    chunk.hash_anchor
  )
)

[GCP CE 핵심 아키텍처 포인트 - The Remote MCP Gap]:
1. 구글 공식 BigQuery Remote MCP(execute_sql_readonly)는 GQL 문법을 파싱하지 못함.
2. 따라서 AI 에이전트가 GQL 쿼리를 날릴 수 있도록 Step 3의 'Tier 3 Domain MCP'가 필수적임.


### 3. 결과 해석 및 실무적 의미 (Interpretation)
- **제로 카피(Zero-Copy)**: DW의 `dw_device_master`에 새로운 스마트폰 모델이 INSERT되더라도 그래프 DB로 복제하는 별도 파이프라인이 필요 없으며, GQL 쿼리가 실시간으로 최신 데이터를 읽습니다.
- **저장소 절감**: 텍스트 중복 엔티티 0건을 달성하여 일반 지식그래프 대비 저장 비용 80% 이상 절감.
- **Step 3 연계**: 이 그래프 구조를 에이전트가 직접 다루지 않고, 안전하게 캡슐화된 11종 MCP 도구를 통해 탐색하도록 Step 3에서 통합합니다.
